# Notebook 02: chuyển giao giữa bốn corpus

Các mô hình học trên PhishVN chấm các corpus khác ra sao, và ngược lại. Notebook này chạy ma trận chuyển giao giữa bốn corpus và các cách sửa, huấn luyện gộp,
khoảng tin cậy bootstrap, và SHAP. Cần chạy notebook 01 trước: các con số ở cuối đọc bảng benchmark của nó.

| Notebook | Nội dung | Đọc kết quả của |
|---|---|---|
| `01_benchmark_and_protocol.ipynb` | benchmark bảy họ, chia theo thời gian và các đối chứng, dòng trùng, tầng nhãn | |
| `02_transfer.ipynb` | ma trận chuyển giao giữa bốn corpus, huấn luyện gộp, khoảng tin cậy, SHAP | 01 |
| `03_robustness.ipynb` | tối ưu siêu tham số, stacking, dự báo độ trôi, nhãn nhiễu | 01 |
| `04_cnn_and_dedup.ipynb` | CNN trên ký tự, bỏ dòng trùng, bỏ tiền tố `www.`, tổng hợp cả bốn notebook | 01, 02 |

## Bước 0: Thiết lập

`setup()` (trong `p2lib/setup.py`) đọc các biến môi trường ở trên, tải bộ PhishVN từ Mendeley Data và ba corpus ngoài từ Zenodo, kiểm mã SHA-256 của từng file,
dựng cây thư mục `url_work/data/processed/` mà mã của bài đọc và ghi, rồi chuyển thư mục làm việc vào đó. Các file đã tải ở notebook trước không tải lại. Mọi kết
quả nằm trong `url_work/data/processed/p2/`, và các notebook sau đọc kết quả của notebook trước từ đó.

In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits

import p2lib
from p2lib import common, transfer, derived
from p2lib.checks import RES, MET_TOL, CURVE_TOL, THR_TOL, ROUND4_TOL, CNN_TOL, REL_TOL, check, check_table, check_matrix, check_json, report

env = p2lib.setup.setup()
p2lib.setup.require(f"{RES}/p2_benchmark.csv", f"{RES}/p2_temporal_strict.csv", notebook="01")

## Bước 1: Ma trận chuyển giao giữa bốn corpus

Bốn corpus (PhishVN, PhiUSIIL, ISCX-URL2016, PhishStorm) đã ở cùng lược đồ 21 đặc trưng. Ô (i, j) huấn luyện trên corpus i và kiểm tra trên corpus j, trung bình
5 seed (`cell_score`). Ô đường chéo kiểm tra trong chính corpus: PhishVN dùng cách chia chính thức của nó; ba corpus kia dùng chia 70/30 theo nhóm tên miền
(`in_dataset_split`), vì ở các corpus đó rất nhiều dòng chung tên miền, và chia ngẫu nhiên từng dòng sẽ để cùng tên miền nằm cả hai phía. Khoảng cách tổng quát
hoá là trung bình đường chéo trừ trung bình ngoài đường chéo.

Bước này chạy 17 ma trận: random forest, hồi quy logistic và HistGB theo F1 và ROC-AUC; CatBoost theo F1; ba họ đó khi bỏ ba đặc trưng mà SHAP chỉ ra là
mang dấu vết riêng của từng corpus (`tld_len`, `subdom_cnt`, `dot_cnt`; arm "pruned"); và random forest, hồi quy logistic với CORAL, cách căn chỉnh phân phối
không giám sát: chuẩn hoá từng corpus rồi biến đổi đặc trưng nguồn cho ma trận hiệp phương sai của nó khớp với corpus đích (`CoralAligner`).

Danh sách 17 ma trận nằm ở `transfer.MATRICES`, mỗi phần tử là tên file kết quả và tham số của lượt chạy.

In [ ]:
for name, kw in transfer.MATRICES:
    transfer.run_cross_dataset(out=f"{RES}/{name}", **kw)
    check_matrix("1 transfer matrices", name)
report("1 transfer matrices")

## Bước 2: Huấn luyện gộp

Liệu huấn luyện trên nhiều corpus cùng lúc có chuyển giao tốt hơn? Với mỗi corpus, random forest được huấn luyện trên hợp của ba corpus còn lại rồi kiểm tra trên
corpus bị giữ ra, với đủ 21 đặc trưng và với bộ đã bỏ ba đặc trưng, rồi so với ô tốt nhất và trung bình của các mô hình huấn luyện trên một corpus (đọc từ ma trận
của Bước 1).

In [ ]:
transfer.run_combined_training()
check_table("2 pooled training", "combined_training.csv", MET_TOL)
report("2 pooled training")

## Bước 3: Khoảng tin cậy của ma trận chuyển giao

Mỗi ô của ma trận là một trung bình trên 5 seed; khoảng tin cậy 95% lấy bằng bootstrap trên các dòng test (500 lần, seed cố định), áp vào trung bình đường chéo,
trung bình ngoài đường chéo, khoảng cách giữa chúng, và từng ô. Hai lượt: CatBoost cho F1, random forest cho ROC-AUC (lượt này ghi thêm từng ô, để biết ô nào
dưới mức ngẫu nhiên 0,5 một cách chắc chắn). Hai bảng này làm tròn bốn chữ số, nên được so với dung sai một đơn vị làm tròn (`ROUND4_TOL`).

In [ ]:
transfer.run_xdata_bootstrap()
check_table("3 transfer bootstrap", "p2_xdata_bootstrap.csv", ROUND4_TOL)
transfer.run_xdata_bootstrap_rf()
check_table("3 transfer bootstrap", "p3_xdata_bootstrap.csv", ROUND4_TOL)
report("3 transfer bootstrap")

## Bước 4: SHAP và trọng số lớp

Đặc trưng nào random forest dựa vào trên PhishVN, và câu trả lời có đổi khi bỏ trọng số lớp không? Với mỗi seed, TreeSHAP tính trên tối đa 2.000 dòng test cân bằng
hai lớp; mỗi đặc trưng có độ lớn đóng góp trung bình, tỷ trọng trong tổng, và hướng (tương quan giữa giá trị đặc trưng và đóng góp của nó).

In [ ]:
transfer.run_shap_weighting()
check_table("4 SHAP", "p2_shap_weighting.csv", MET_TOL)
report("4 SHAP")

## Bước 5: Các con số tính từ kết quả

Phần lớn con số của bài là trung bình và độ lệch đọc thẳng từ các bảng đã so ở trên. Một số khác được tính thêm từ các bảng đó hoặc từ chính corpus: kiểm định
có cặp, bootstrap, tương quan, và vài phép đếm lại. `derived.transfer_numbers` (trong `p2lib/derived.py`) tính lại những con số thuộc các bảng của notebook này và so
từng nhóm với `results/p2_generated.json`, file bài ghi các con số đó ra. Mỗi số được coi là khớp nếu lệch không quá `MET_TOL`
hoặc không quá `REL_TOL` = 10⁻³ tính tương đối: thống kê t chia cho một sai số chuẩn nhỏ, nên chữ số cuối lệch do các họ boosting cộng song song bị phóng
to ở đó (bài không in thống kê t, chỉ in hiệu và p). Bốn notebook cùng phủ hết các nhóm của file đó; notebook 04 kiểm lại điều này.

In [ ]:
R = derived.transfer_numbers(common.load())
for key in R:
    check_json("5 derived numbers", R[key], "results/p2_generated.json", key=key, tol=MET_TOL, rel_tol=REL_TOL)
report("5 derived numbers")

## Bước 6: Tổng hợp

Bảng dưới gom mọi phép kiểm của notebook này theo bước, và lưu chúng vào `data/processed/p2/notebook_checks/` để notebook 04 cộng cả bốn.

In [ ]:
p2lib.checks.summary("02")